In [1]:
import os, sys, shutil
import numpy as np
import pandas as sp
import pandas as pd
import nibabel as nib
import json

In [2]:
home = os.path.expanduser("~")
print(home)
atlas_dir = os.path.join(home, 'research_projects/snaplab_tools/data/atlases')

/home/lindenmp


## MNIvolumetric

In [3]:
# MNIvolumetric
in_dir = os.path.join(home, 'MSA/subcortex/Group-Parcellation/3T/Cortex-Subcortex/MNIvolumetric')

# MNI152NLin2009cAsym
for n_regions in [100, 200, 400]:
    for tian_scale in [1, 2, 3, 4]:
        out_dir = os.path.join(atlas_dir, 'SchaeferMSA/atlas-Schaefer{0}7MSA{1}'.format(n_regions, tian_scale))
        if os.path.isdir(out_dir):
            shutil.rmtree(out_dir)
        os.makedirs(out_dir)
        for res in [1, 2]:
            shutil.copyfile(
                os.path.join(in_dir, 'Schaefer2018_{0}Parcels_7Networks_order_Tian_Subcortex_S{1}_3T_MNI152NLin2009cAsym_{2}mm.nii.gz'.format(n_regions, tian_scale, res)),
                os.path.join(out_dir, 'atlas-Schaefer{0}7MSA{1}_space-MNI152NLin2009cAsym_res-0{2}_dseg.nii.gz'.format(n_regions, tian_scale, res))
                )

# MNI152NLin6Asym
for n_regions in [100, 200, 400]:
    for tian_scale in [1, 2, 3, 4]:
        out_dir = os.path.join(atlas_dir, 'SchaeferMSA/atlas-Schaefer{0}7MSA{1}'.format(n_regions, tian_scale))
        for res in [1, 2]:
            shutil.copyfile(
                os.path.join(in_dir, 'Schaefer2018_{0}Parcels_7Networks_order_Tian_Subcortex_S{1}_MNI152NLin6Asym_{2}mm.nii.gz'.format(n_regions, tian_scale, res)),
                os.path.join(out_dir, 'atlas-Schaefer{0}7MSA{1}_space-MNI152NLin6Asym_res-0{2}_dseg.nii.gz'.format(n_regions, tian_scale, res))
                )

## Support text files

In [4]:
def process_tsv(tsv_file):
    df = pd.read_csv(tsv_file, header=None)
    idx_filter = np.arange(0, df.shape[0], 2)
    df = df.loc[idx_filter]
    df.reset_index(inplace=True, drop=True)
    df.index = df.index + 1
    df.index.name = 'index'
    df.rename(columns={0: 'label'}, inplace=True)
    
    for i in np.arange(df.shape[0]):
        if '7Networks' in df.loc[i + 1, 'label']:
            df.loc[i + 1, 'cortex'] = True
        else:
            df.loc[i + 1, 'cortex'] = False
    
    df.to_csv(tsv_file, sep="\t")

In [5]:
# copy
in_dir = os.path.join(home, 'MSA/subcortex/Group-Parcellation/3T/Cortex-Subcortex')

for n_regions in [100, 200, 400]:
    for tian_scale in [1, 2, 3, 4]:
        out_dir = os.path.join(atlas_dir, 'SchaeferMSA/atlas-Schaefer{0}7MSA{1}'.format(n_regions, tian_scale))
        tsv_file = os.path.join(out_dir, 'atlas-Schaefer{0}7MSA{1}_dseg.tsv'.format(n_regions, tian_scale))

        shutil.copyfile(
            os.path.join(in_dir, 'Schaefer2018_{0}Parcels_7Networks_order_Tian_Subcortex_S{1}_label.txt'.format(n_regions, tian_scale)),
            tsv_file
            )
        
        process_tsv(tsv_file)

        json_file = os.path.join(out_dir, 'atlas-Schaefer{0}7MSA{1}_dseg.json'.format(n_regions, tian_scale))
        data = {"BIDSVersion": "1.8.0", "Name": "Schaefer{0}7MSA{1}".format(n_regions, tian_scale)}
        # creating a JSON string
        json_string = json.dumps(data)
        # storing it in a file
        with open(json_file, "w") as json_data:
            json.dump(data, json_data)

## Swap cortex/subcortex order

In [6]:
for n_regions in [100, 200, 400]:
    for tian_scale in [1, 2, 3, 4]:
        in_dir = os.path.join(atlas_dir, 'SchaeferMSA/atlas-Schaefer{0}7MSA{1}'.format(n_regions, tian_scale))
        tsv_file = os.path.join(in_dir, 'atlas-Schaefer{0}7MSA{1}_dseg.tsv'.format(n_regions, tian_scale))
        
        # create remapping index in df
        df = pd.read_csv(tsv_file, header=0, index_col=0, sep='\t')
        sort_idx = np.concatenate((df.index[df['cortex'] == True].values[:, np.newaxis],
                                    df.index[df['cortex'] != True].values[:, np.newaxis]), axis = 0)
        sort_idx = np.squeeze(sort_idx)
        df = df.loc[sort_idx, :]
        df['subcortex'] = df['cortex'] == False
        df['new_index'] = np.arange(1, df.shape[0] + 1)

        for space in ['MNI152NLin2009cAsym', 'MNI152NLin6Asym']:
            for res in [1, 2]:
                print(n_regions, tian_scale, space, res)
                parc_file = os.path.join(in_dir, 'atlas-Schaefer{0}7MSA{1}_space-{2}_res-0{3}_dseg.nii.gz'.format(n_regions, tian_scale, space, res))

                # load parc file
                parc = nib.load(parc_file)
                parc_data = parc.get_fdata()
                parc_data_new = np.zeros(parc_data.shape)
                
                # write new indices to new parc data
                for i in np.arange(df.shape[0]):
                    roi_mask = parc_data == df.index[i]
                    parc_data_new[roi_mask] = df.iloc[i]['new_index']
                
                # save out (overwrite)
                parc_out = nib.Nifti1Image(parc_data_new, affine=parc.affine, header=parc.header)
                nib.save(parc_out, parc_file)

        df.set_index('new_index', inplace=True)
        df.index.name = 'index'
        df.to_csv(tsv_file, sep="\t")


100 1 MNI152NLin2009cAsym 1
100 1 MNI152NLin2009cAsym 2
100 1 MNI152NLin6Asym 1
100 1 MNI152NLin6Asym 2
100 2 MNI152NLin2009cAsym 1
100 2 MNI152NLin2009cAsym 2
100 2 MNI152NLin6Asym 1
100 2 MNI152NLin6Asym 2
100 3 MNI152NLin2009cAsym 1
100 3 MNI152NLin2009cAsym 2
100 3 MNI152NLin6Asym 1
100 3 MNI152NLin6Asym 2
100 4 MNI152NLin2009cAsym 1
100 4 MNI152NLin2009cAsym 2
100 4 MNI152NLin6Asym 1
100 4 MNI152NLin6Asym 2
200 1 MNI152NLin2009cAsym 1
200 1 MNI152NLin2009cAsym 2
200 1 MNI152NLin6Asym 1
200 1 MNI152NLin6Asym 2
200 2 MNI152NLin2009cAsym 1
200 2 MNI152NLin2009cAsym 2
200 2 MNI152NLin6Asym 1
200 2 MNI152NLin6Asym 2
200 3 MNI152NLin2009cAsym 1
200 3 MNI152NLin2009cAsym 2
200 3 MNI152NLin6Asym 1
200 3 MNI152NLin6Asym 2
200 4 MNI152NLin2009cAsym 1
200 4 MNI152NLin2009cAsym 2
200 4 MNI152NLin6Asym 1
200 4 MNI152NLin6Asym 2
400 1 MNI152NLin2009cAsym 1
400 1 MNI152NLin2009cAsym 2
400 1 MNI152NLin6Asym 1
400 1 MNI152NLin6Asym 2
400 2 MNI152NLin2009cAsym 1
400 2 MNI152NLin2009cAsym 2
400 2 MN

## Copy dataset_description

In [7]:
in_file = os.path.join(atlas_dir, 'dataset_description.json')
out_file = os.path.join(atlas_dir, 'SchaeferMSA', 'dataset_description.json')
shutil.copyfile(in_file, out_file)

'/home/lindenmp/research_projects/snaplab_tools/data/atlases/SchaeferMSA/dataset_description.json'